In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [3]:
# ── Load EDA Data ─────────────────────────────────────────
df = pd.read_csv('../data/processed/eda_data.csv')
print(f"Loaded: {df.shape[0]:,} rows | {df.shape[1]:,} columns")

target = 'isFraud'
original_shape = df.shape

Loaded: 590,540 rows | 436 columns


In [4]:
# ============================================================
# STEP 1: DROP HIGH-MISSING COLUMNS (>70% missing)
# Reason: No imputation strategy can recover a column
#         that is 87-99% empty — it adds noise, not signal
# ============================================================

missing_pct = df.isnull().mean()
drop_cols   = missing_pct[missing_pct > 0.70].index.tolist()

# Never drop target
drop_cols = [c for c in drop_cols if c != target]

df.drop(columns=drop_cols, inplace=True)

print(f"\n[STEP 1] Dropped {len(drop_cols)} columns with >70% missing")
print(f"Shape after drop: {df.shape}")


[STEP 1] Dropped 208 columns with >70% missing
Shape after drop: (590540, 228)


In [5]:
# ============================================================
# STEP 2: SEPARATE COLUMN TYPES
# Reason: Numeric and categorical need different
#         imputation strategies
# ============================================================

# Exclude ID and target from processing
exclude_cols = ['TransactionID', target]

numeric_cols     = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

numeric_cols     = [c for c in numeric_cols if c not in exclude_cols]
categorical_cols = [c for c in categorical_cols if c not in exclude_cols]

print(f"\n[STEP 2] Column Types After Drop")
print(f"  Numeric columns    : {len(numeric_cols)}")
print(f"  Categorical columns: {len(categorical_cols)}")


[STEP 2] Column Types After Drop
  Numeric columns    : 213
  Categorical columns: 13


In [6]:
# ============================================================
# STEP 3: IMPUTE NUMERIC COLUMNS
# Strategy:
#   - TransactionAmt → no missing, skip
#   - C* columns (counts) → fill with 0 (absence = 0 count)
#   - D* columns (time deltas) → fill with median
#   - V* columns (Vesta engineered) → fill with median
#   - All others → fill with median
# Reason: Mean is sensitive to outliers (fraud transactions
#         skew distributions significantly)
# ============================================================

# Identify column groups
c_cols = [c for c in numeric_cols if c.startswith('C')]
d_cols = [c for c in numeric_cols if c.startswith('D')]
v_cols = [c for c in numeric_cols if c.startswith('V')]
m_cols = [c for c in numeric_cols if c.startswith('M')]

print(f"\n[STEP 3] Numeric Column Groups")
print(f"  C* (count features)  : {len(c_cols)}")
print(f"  D* (time deltas)     : {len(d_cols)}")
print(f"  V* (Vesta features)  : {len(v_cols)}")
print(f"  M* (match features)  : {len(m_cols)}")

# C* → fill with 0
for col in c_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(0, inplace=True)

# D*, V* and remaining numeric → fill with median
remaining_numeric = [c for c in numeric_cols if c not in c_cols]
for col in remaining_numeric:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

# Verify
remaining_numeric_missing = df[numeric_cols].isnull().sum().sum()
print(f"\n  Missing in numeric after imputation: {remaining_numeric_missing}")


[STEP 3] Numeric Column Groups
  C* (count features)  : 14
  D* (time deltas)     : 8
  V* (Vesta features)  : 180
  M* (match features)  : 0

  Missing in numeric after imputation: 0


In [7]:
# ============================================================
# STEP 4: IMPUTE CATEGORICAL COLUMNS
# Strategy: Fill with 'unknown'
# Reason:
#   - Missing email domain / card info is itself a fraud signal
#   - Filling with mode hides that signal
#   - 'unknown' preserves the missingness as a category
# ============================================================

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna('unknown', inplace=True)

remaining_cat_missing = df[categorical_cols].isnull().sum().sum()
print(f"\n[STEP 4] Missing in categorical after imputation: {remaining_cat_missing}")



[STEP 4] Missing in categorical after imputation: 0


In [8]:
# ============================================================
# STEP 5: ENCODE CATEGORICAL COLUMNS
# Strategy:
#   - Low cardinality (<= 10 unique) → Label Encoding
#   - High cardinality (> 10 unique) → Frequency Encoding
# Reason:
#   - One-hot encoding on high-cardinality cols like
#     email domain would explode dimensionality
#   - Frequency encoding converts category to its
#     occurrence rate — preserves signal, controls size
# ============================================================

print(f"\n[STEP 5] Encoding Categorical Columns")

label_encoded_cols = []
freq_encoded_cols  = []

for col in categorical_cols:
    n_unique = df[col].nunique()

    if n_unique <= 10:
        # Label Encoding
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoded_cols.append(col)
    else:
        # Frequency Encoding
        freq_map = df[col].value_counts(normalize=True).to_dict()
        df[col]  = df[col].map(freq_map)
        freq_encoded_cols.append(col)

print(f"  Label encoded  : {len(label_encoded_cols)} columns → {label_encoded_cols}")
print(f"  Freq encoded   : {len(freq_encoded_cols)} columns → {freq_encoded_cols}")



[STEP 5] Encoding Categorical Columns
  Label encoded  : 12 columns → ['ProductCD', 'card4', 'card6', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9']
  Freq encoded   : 1 columns → ['P_emaildomain']


In [9]:
# ============================================================
# STEP 6: TRANSACTION AMOUNT — LOG TRANSFORMATION
# Reason:
#   - TransactionAmt is heavily right-skewed
#   - EDA showed outliers up to very high values
#   - Log transform compresses scale, helps tree models
#     and is essential if we add linear models later
# ============================================================

df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])

print(f"\n[STEP 6] Log-transformed TransactionAmt")
print(f"  Original  - Mean: {df['TransactionAmt'].mean():.2f} | Std: {df['TransactionAmt'].std():.2f}")
print(f"  Log       - Mean: {df['TransactionAmt_log'].mean():.2f} | Std: {df['TransactionAmt_log'].std():.2f}")


[STEP 6] Log-transformed TransactionAmt
  Original  - Mean: 135.03 | Std: 239.16
  Log       - Mean: 4.38 | Std: 0.94


In [10]:
# ============================================================
# STEP 7: DROP LOW-VARIANCE COLUMNS
# Reason:
#   - Columns where 99%+ rows have same value add no
#     predictive power and slow down training
# ============================================================

low_variance_cols = []
for col in df.columns:
    if col in exclude_cols + [target]:
        continue
    top_freq = df[col].value_counts(normalize=True).iloc[0]
    if top_freq >= 0.99:
        low_variance_cols.append(col)

df.drop(columns=low_variance_cols, inplace=True)

print(f"\n[STEP 7] Dropped {len(low_variance_cols)} low-variance columns")
print(f"Shape after low-variance drop: {df.shape}")


[STEP 7] Dropped 25 low-variance columns
Shape after low-variance drop: (590540, 204)


In [11]:
# ============================================================
# STEP 8: DROP DUPLICATE ROWS
# Reason: Duplicate transactions could bias model training
# ============================================================

before = len(df)
df.drop_duplicates(inplace=True)
after  = len(df)

print(f"\n[STEP 8] Duplicate rows removed: {before - after:,}")
print(f"Shape after dedup: {df.shape}")


[STEP 8] Duplicate rows removed: 0
Shape after dedup: (590540, 204)


In [ ]:
# ============================================================
# STEP 9: FINAL VALIDATION
# ============================================================

total_missing  = df.isnull().sum().sum()
total_cols     = df.shape[1]
numeric_final  = df.select_dtypes(include=[np.number]).shape[1]

print(f"""
========== PREPROCESSING SUMMARY ==========
Original Shape         : {original_shape}
Final Shape            : {df.shape}
Columns Dropped (70%+) : {len(drop_cols)}
Low Variance Dropped   : {len(low_variance_cols)}
Remaining Missing      : {total_missing}
Numeric Columns        : {numeric_final}
Label Encoded          : {len(label_encoded_cols)}
Freq Encoded           : {len(freq_encoded_cols)}
============================================
""")




========== PREPROCESSING SUMMARY ==========
Original Shape         : (590540, 436)
Final Shape            : (590540, 204)
Columns Dropped (70%+) : 208
Low Variance Dropped   : 25
Remaining Missing      : 0
Numeric Columns        : 204
Label Encoded          : 12
Freq Encoded           : 1



In [15]:
# ============================================================
# STEP 10: SAVE CLEAN DATA
# ============================================================

df.to_csv('../data/processed/clean_data.csv', index=False)
print(f"\nClean data saved → data/processed/clean_data.csv")


Clean data saved → data/processed/clean_data.csv
